# 03 SFT：从可验证轨迹到规则记忆与执行

本节研究监督数据和推理轨迹如何让 `Qwen/Qwen3-0.6B` 记忆、检索并执行课程提供的规则。核心对照是 Direct Full-Trace、Joint Drill + Full-Trace 和 Two-stage Memorization → Execution；LoRA / QLoRA 是同一 Notebook 内的参数高效分支。

当前 DGX Spark 阶段只进行数据审计、Label Masking、Loss 和小模型 CPU Smoke Test。模型加载、训练和显存测量必须在明天的 RTX 5090 官方 Profile 中执行；本 Notebook 默认 `RUN_SFT=False`。

当前默认基座为 `Qwen/Qwen3-0.6B`，保留官方 Thinking 模板。原 GPU Baseline 与 SFT/DPO/PPO/GRPO 资源 Smoke 已完成；新版低显存实现已通过本地 CPU 检查，尚待 GPU 复测。见 [基准报告](docs/QWEN3_0_6B_BENCHMARK.md) 与 [低显存说明](docs/QWEN3_0_6B_LOW_MEMORY.md)。

本次仅切换模型配置；本节仍使用旧版 Gravity / Equation Numeric / Bit 数据接口，尚未迁移到新版 Numeral 三任务。


## 学习目标

- 判断教师轨迹是否能从 Prompt 和公开先验推导；
- 将 Prompt Token Mask 为 `-100`，只对 Assistant Completion 计算 Cross Entropy；
- 比较 Direct、Joint 和 Two-stage 的训练数据与 Token Budget；
- 理解 BF16 LoRA 与 NF4 QLoRA 的工程边界；
- 使用自定义 PyTorch 训练循环保存可交接的 Adapter 和日志。

## 1. 实验控制与数据边界

真实数据划分来自 `00_Baseline_Diagnosis.ipynb`。默认 **Real Only**：读取维护者预先用原有 Solver 和独立 Verifier
准备的真实 Train 参考轨迹 `data/processed/real_train_traces.jsonl`，只保留当前 Train 中的原始 ID。
真实 Prompt 不改写，原始答案只在求解后用于核对，不用于反向构造推理。

`01_Synthetic_Data_Engineering.ipynb` 完全可选。只有显式设置 `USE_SYNTHETIC_DATA=1`，
才追加 `synthetic/audited/synthetic_accepted.jsonl`。此时校验 Accepted 哈希、独立 Verifier 状态、
三集合泄漏审计和当前划分指纹；文件缺失、未经审计或划分改变均明确报错，不回退到历史 Accepted。

| 策略 | 训练内容 | 研究问题 |
|---|---|---|
| SFT-A Direct | 完整可验证轨迹 | 直接学习规则执行 |
| SFT-B Joint | Recall Drill + 完整轨迹混合 | 记忆与执行是否互补 |
| SFT-C Two-stage | 先 Drill，再 Full Trace | Memorization → Execution 是否有序收益 |
| SFT-D Targeted Drill | 从错误分析选择局部 Drill | 选做，不无差别重复 |

比较时记录训练 Token Budget、更新步数和数据来源；两阶段多跑的 Token 必须单独报告，不能把训练量增加误认为顺序收益。

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from dataclasses import dataclass
from collections import Counter
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

project_root = Path.cwd()
synthetic_accepted_path = project_root / "data" / "processed" / "synthetic" / "audited" / "synthetic_accepted.jsonl"
real_trace_path = project_root / "data" / "processed" / "real_train_traces.jsonl"
use_synthetic_data = os.environ.get("USE_SYNTHETIC_DATA", "0") == "1"
train_path = project_root / "data" / "processed" / "splits" / "train.csv"
result_dir = project_root / "data" / "processed" / "sft"
checkpoint_root = project_root / "data" / "processed" / "checkpoints"
MODEL_ID = os.environ.get("QWEN3_MODEL_ID", "Qwen/Qwen3-0.6B")
MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1].lower().replace(".", "_")
DEFAULT_MODEL_PATH = Path.cwd() / "models" / MODEL_ID.rsplit("/", 1)[-1]
if not DEFAULT_MODEL_PATH.is_dir():
    DEFAULT_MODEL_PATH = Path.home() / "autodl-tmp/huggingface/models" / MODEL_ID.rsplit("/", 1)[-1]
MODEL_SOURCE = os.environ.get("QWEN3_MODEL_PATH", os.environ.get("LOCAL_MODEL_PATH",
    str(DEFAULT_MODEL_PATH) if DEFAULT_MODEL_PATH.is_dir() else MODEL_ID))
# Catch a stale 4B path before recording it under the 0.6B model ID.
if MODEL_ID == "Qwen/Qwen3-0.6B" and Path(MODEL_SOURCE).is_dir():
    _local_config = json.loads((Path(MODEL_SOURCE) / "config.json").read_text())
    if (_local_config.get("model_type"), _local_config.get("hidden_size"), _local_config.get("num_hidden_layers")) != ("qwen3", 1024, 28):
        raise ValueError("QWEN3_MODEL_PATH 与 Qwen3-0.6B 不匹配；请检查旧模型环境变量")
result_dir = result_dir / MODEL_TAG / os.environ.get("GPU_SMOKE_RUN_ID", "")
checkpoint_root = checkpoint_root / MODEL_TAG / os.environ.get("GPU_SMOKE_RUN_ID", "")
sft_manifest_path = checkpoint_root / "sft_manifest.json"
base_model_path = MODEL_SOURCE
tokenizer_path = os.environ.get("QWEN3_TOKENIZER_PATH", base_model_path)
SEED = 42
run_sft = os.environ.get("RUN_SFT", "0") == "1"
run_mode = os.environ.get("RUN_MODE", "smoke")       # smoke / full
max_length = int(os.environ.get("SFT_MAX_LENGTH", "1024"))
MICRO_BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
max_train_records = 24 if run_mode == "smoke" else 512
SMOKE_QUANTIZATION_MODE = "nf4_qlora"
TRAINING_PROFILE = "official-q-qlora-or-pt-sft-cuda"
# CPU mechanism tests are deliberately pinned to CPU even when a CUDA device is visible.
device = "cpu"

if not train_path.exists():
    raise FileNotFoundError("缺少 00_Baseline_Diagnosis.ipynb 生成的 Train 划分")
train_df = pd.read_csv(train_path, dtype=str, keep_default_na=False)

print({
    "model_id": MODEL_ID,
    "device_for_cpu_smoke": device,
    "run_sft": run_sft,
    "run_mode": run_mode,
    "max_length": max_length,
    "train_rows": len(train_df),
})

## 2. Prompt-derivable Teacher Trace

默认读取真实 Train 的已验证参考轨迹；显式启用增强后再混入 Accepted。两类数据沿用相同的
Full Trace 格式、Trace Audit 和 SFT Loss。`hidden_rule` 元数据不进入 Prompt 或 Completion。


### 2.1 读取真实 Train；按开关读取 Accepted


In [ ]:
def load_real_train_records() -> list[dict[str, object]]:
    manifest = json.loads(real_trace_path.with_suffix(".manifest.json").read_text(encoding="utf-8"))
    if hashlib.sha256(real_trace_path.read_bytes()).hexdigest() != manifest["sha256"]:
        raise ValueError("维护者提供的真实轨迹文件与 Manifest 不一致")
    samples = [json.loads(line) for line in real_trace_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    train_by_id = train_df.set_index("id").to_dict(orient="index")
    selected = []
    for sample in samples:
        original = train_by_id.get(str(sample.get("source_id", "")))
        if original is None:
            continue
        if sample["prompt"] != original["prompt"] or sample["task_type"] != original["task_type"] or sample["original_answer"] != original["answer"]:
            raise ValueError("真实轨迹与当前 Train 原题不一致")
        if sample.get("verification_status") != "accepted" or not sample.get("independent_verification_passed"):
            raise ValueError("真实轨迹未通过独立验证")
        selected.append(sample)
    if not selected:
        raise ValueError("当前 Train 没有可用的真实参考轨迹，请维护者核对数据版本")
    return selected


def load_screened_synthetic_records(path: Path = synthetic_accepted_path) -> list[dict[str, object]]:
    if not use_synthetic_data:
        return []
    # 只有显式开启增强，才接触合成文件和审计产物。
    manifest = json.loads(path.with_name("accepted_manifest.json").read_text(encoding="utf-8"))
    dataset_path = project_root / "data" / "processed" / "dataset_manifest.json"
    if manifest.get("status") != "passed" or not manifest.get("all_accepted_verified_and_audited"):
        raise ValueError("Accepted Manifest 未通过独立验证和泄漏审计")
    if hashlib.sha256(path.read_bytes()).hexdigest() != manifest["accepted_sha256"]:
        raise ValueError("Accepted 内容已变化，需重新运行 01 审计")
    if hashlib.sha256(dataset_path.read_bytes()).hexdigest() != manifest["dataset_manifest_sha256"]:
        raise ValueError("真实 Dataset Manifest 已变化，需重新运行 01 审计")
    audit_path = path.with_name("synthetic_real_leakage_audit.json")
    if hashlib.sha256(audit_path.read_bytes()).hexdigest() != manifest["audit_sha256"]:
        raise ValueError("Leakage Audit Report 已变化，需重新审计")
    for name, expected in manifest["split_sha256"].items():
        split_path = project_root / "data" / "processed" / "splits" / f"{name}.csv"
        if hashlib.sha256(split_path.read_bytes()).hexdigest() != expected:
            raise ValueError(f"{name} 划分已变化，需重新运行 01 审计")
    if set(manifest["split_sha256"]) != {"train", "validation", "holdout_test"}:
        raise ValueError("Accepted 缺少三个集合的审计指纹")
    samples = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(samples) != manifest["accepted_rows"]:
        raise ValueError("Accepted 行数与 Manifest 不一致")
    for sample in samples:
        if (sample.get("verification_status") != "accepted" or sample.get("leakage_status") != "passed"
                or sample.get("verifier_version") != manifest["verifier_version"]):
            raise ValueError("合成样本缺少独立 Verifier 或 Leakage Audit 通过记录")
    return samples


### 2.2 轨迹格式：把公开规则变成可审计文本

In [ ]:
def _format_trace(sample: dict[str, object]) -> str:
    task = str(sample.get("task_type", ""))
    trace = sample.get("trace", {}) if isinstance(sample.get("trace", {}), dict) else {}
    answer = str(sample.get("answer", "")).strip()
    if task == "numeral_system":
        reasoning = ("1. Check the standard Roman rules against the public examples.\n"
                     f"2. Decompose {trace.get('query_number')} into {trace.get('components')}.\n"
                     "3. Concatenate the place-value components.")
    elif task == "gravity":
        g = trace.get("g_ols", "inferred from examples")
        query_t = trace.get("query_t", "the Query time")
        query_raw = trace.get("query_raw", "computed from the inferred g")
        reasoning = (
            "1. Parse all public time-distance examples.\n"
            "2. Fit g = 2*sum(t^2*d)/sum(t^4) using only those examples.\n"
            f"3. The fitted g is {g}; apply distance = 0.5*g*t^2 to Query t={query_t}.\n"
            f"4. The unrounded Query distance is {query_raw}; format the final numeric answer."
        )
    elif task == "equation_numeric":
        candidate_rules = trace.get("candidate_rules_by_operator", {})
        outputs = trace.get("query_outputs", [])
        reasoning = (
            "1. Parse every AB<operator>CD example.\n"
            f"2. Keep only public 24-rule candidates consistent with each example: {candidate_rules}.\n"
            f"3. Apply the normal/flip family and semantic-group constraints; valid assignment count is {trace.get('valid_assignment_count', 'unknown')}.\n"
            f"4. Apply the selected candidates to the Query; valid Query outputs are {outputs}.\n"
            "5. Keep the answer only when the verifier finds one unique Query output."
        )
    elif task == "bit_manipulation":
        ident = trace.get("identifiability", {})
        expression = trace.get("expression", "a candidate expression inferred from examples")
        reasoning = (
            "1. Parse all 8-bit input-output examples.\n"
            "2. Enumerate legal DSL expressions and discard candidates that fail any example.\n"
            f"3. The selected semantic candidate is {expression}; identifiability summary is {ident}.\n"
            "4. Apply that candidate to the Query and format exactly eight bits."
        )
    else:
        reasoning = "1. Use the public task specification.\n2. Verify every example before computing the Query."
    return reasoning + f"\nFinal answer: \\boxed{{{answer}}}"


### 2.3 构造与审核单条 Teacher Trace

In [ ]:
def make_trace_record(sample: dict[str, object]) -> dict[str, object]:
    completion = _format_trace(sample)
    return {
        "prompt": str(sample.get("prompt", "")),
        "reasoning": completion.rsplit("\nFinal answer:", 1)[0],
        "final_answer": str(sample.get("answer", "")).strip(),
        "completion": completion,
        "task_type": str(sample.get("task_type", "")),
        "trace_type": "full_trace",
        "source": "screened_synthetic_train_prior" if sample.get("source") == "deterministic_prior_generator" else "verified_real_train",
        "source_id": str(sample.get("source_id", sample.get("candidate_id", ""))),
        "verification_status": str(sample.get("verification_status", "")),
        "hidden_rule_used_in_completion": False,
        "token_length_proxy": len(completion.split()),
    }


def audit_trace_record(record: dict[str, object], max_tokens: int = max_length) -> dict[str, object]:
    completion = str(record.get("completion", ""))
    reasoning = str(record.get("reasoning", ""))
    answer = str(record.get("final_answer", "")).strip()
    checks = {
        "schema_ok": all(record.get(field) is not None for field in ("prompt", "completion", "task_type", "source_id")),
        "final_answer_present": bool(answer) and f"{answer}}}" in completion,
        "trace_nonempty": bool(reasoning.strip()),
        "prompt_derivable": "hidden_rule" not in completion.lower() and "ground truth" not in completion.lower(),
        "verification_status_ok": record.get("verification_status") == "accepted",
        "length_ok": int(record.get("token_length_proxy", len(completion.split()))) <= max_tokens,
        "task_supported": record.get("task_type") in {"numeral_system", "gravity", "equation_numeric", "bit_manipulation"},
    }
    return {"accepted": all(checks.values()), "checks": checks, "rejection_reasons": [name for name, ok in checks.items() if not ok]}


### 2.4 批量构造 Full Trace 并记录审计结果

In [ ]:
def build_full_trace_records(limit: int | None = None) -> list[dict[str, object]]:
    real_samples = load_real_train_records()
    synthetic_samples = load_screened_synthetic_records() if use_synthetic_data else []
    # 在每个题型内交错来源，让 Smoke 也能覆盖三类增强样本。
    samples = []
    for task in ("gravity", "equation_numeric", "bit_manipulation"):
        real_task = [sample for sample in real_samples if sample["task_type"] == task]
        synthetic_task = [sample for sample in synthetic_samples if sample["task_type"] == task]
        for index in range(max(len(real_task), len(synthetic_task))):
            if index < len(real_task):
                samples.append(real_task[index])
            if index < len(synthetic_task):
                samples.append(synthetic_task[index])
    if limit is not None and samples:
        # Stratify from the complete verified real/accepted data before applying the limit;
        # slicing first would silently produce Gravity-only Smoke data.
        task_order = ("gravity", "equation_numeric", "bit_manipulation")
        grouped = {task: [sample for sample in samples if sample.get("task_type") == task] for task in task_order}
        target = int(limit)
        base = target // len(task_order)
        remainder = target % len(task_order)
        selected = []
        for index, task in enumerate(task_order):
            take = base + (1 if index < remainder else 0)
            selected.extend(grouped[task][:take])
        samples = selected
    elif limit is not None:
        samples = samples[:limit]
    records = [make_trace_record(sample) for sample in samples]
    audited = []
    for record in records:
        audit = audit_trace_record(record)
        record["trace_audit"] = audit
        if audit["accepted"]:
            audited.append(record)
    return audited

full_trace_records = build_full_trace_records(max_train_records)
print({
    "use_synthetic_data": use_synthetic_data,
    "sources": dict(Counter(row["source"] for row in full_trace_records)),
    "full_trace_records_after_audit": len(full_trace_records),
    "by_task": pd.Series([row["task_type"] for row in full_trace_records]).value_counts().to_dict() if full_trace_records else {},
})


## 3. Recall Drill、Joint 和 Two-stage 数据

Recall Drill 只训练公开规则定义、解析模式和格式约定，不包含某道题的隐藏规则。Joint 将 Drill 与完整轨迹混合；Two-stage 先只优化 Drill，再只优化完整轨迹。下面的计数和 Token 代理用于公平记录，不代表实际 GPU 训练结果。

In [ ]:
EQUATION_DRILL_ITEMS = [
    ("equation_parse", "AB_CD parses AB and CD directly; BA_DC reverses both two-digit operands."),
    ("equation_global_constraint", "Non-join rules share one normal or flip family and use at most one rule per semantic group."),
    ("equation_guess", "A missing Query operator is Guess; retain it only when all valid assignments agree on the Query output."),
    ("equation_join", "join concatenates AB then CD; join_swap concatenates CD then AB."),
]
BIT_DRILL_ITEMS = [
    ("bit_mask", "Every Bit Manipulation result is masked to an unsigned 8-bit value."),
    ("bit_rotate", "ROTL and ROTR rotate within eight bits; a shift amount is in 0..7."),
    ("bit_boolean", "AND, OR, XOR and NOT operate on the 8-bit expression value."),
]
GRAVITY_DRILL_ITEMS = [
    ("gravity_formula", "distance = 0.5*g*t^2, with t in seconds, distance in meters, and g inferred per question."),
    ("gravity_ols", "g can be estimated from public examples as 2*sum(t^2*d)/sum(t^4)."),
]


def build_recall_drill_records() -> list[dict[str, object]]:
    items = []
    for task, pairs in (("equation_numeric", EQUATION_DRILL_ITEMS), ("bit_manipulation", BIT_DRILL_ITEMS), ("gravity", GRAVITY_DRILL_ITEMS)):
        for name, statement in pairs:
            items.append({
                "prompt": f"Recall Drill ({task}): state the public rule for {name}.",
                "reasoning": "Recall only the course-provided prior; do not infer a sample-specific hidden rule.",
                "final_answer": statement,
                "completion": f"Rule: {statement}",
                "task_type": task,
                "trace_type": "recall_drill",
                "source": "course_prior_specification",
                "source_id": name,
                "verification_status": "accepted",
                "hidden_rule_used_in_completion": False,
                "token_length_proxy": len(statement.split()),
            })
    return items


def choose_sft_records(strategy: str, limit: int = max_train_records) -> list[dict[str, object]]:
    drills = build_recall_drill_records()
    full = full_trace_records
    if strategy in {"direct", "execution"}:
        return full[:limit]
    if strategy == "memorization":
        return drills[:limit]
    if strategy in {"joint", "two_stage"}:
        # Keep the Smoke budget fixed at 24 without duplicating records.
        full_budget = max(0, int(limit) - len(drills))
        # Re-stratify after reserving the Recall budget; slicing grouped full
        # records here would remove Bit from the 24-record Two-stage Smoke.
        task_order = ("gravity", "equation_numeric", "bit_manipulation")
        selected = []
        for index, task in enumerate(task_order):
            take = full_budget // 3 + (index < full_budget % 3)
            selected.extend([r for r in full if r["task_type"] == task][:take])
        return drills[:limit] + selected
    raise ValueError(f"unknown_sft_strategy:{strategy}")

def summarize_sft_data(strategy: str) -> dict[str, object]:
    records = choose_sft_records(strategy)
    by_type = Counter(str(row["trace_type"]) for row in records)
    return {
        "strategy": strategy,
        "records": len(records),
        "trace_type_counts": dict(by_type),
        "token_budget_proxy": int(sum(int(row["token_length_proxy"]) for row in records)),
        "task_counts": dict(Counter(str(row["task_type"]) for row in records)),
    }


sft_data_summaries = [summarize_sft_data(name) for name in ("direct", "joint", "memorization", "execution", "two_stage")]
print(json.dumps(sft_data_summaries, ensure_ascii=False, indent=2))

## 4. Chat Template、Label Masking 和 SFT Loss

训练样本由 User Prompt 和 Assistant Completion 构成。使用 Tokenizer 官方 Chat Template 生成两段序列，再用前缀长度定位 Completion；不手工拼接 Qwen 特殊 Token。`labels` 中 Prompt 和 Padding 均为 `-100`，因此 Cross Entropy 只看 Assistant Token。

### 4.1 Chat Template 前缀与 Assistant Label Mask

In [ ]:
def encode_chat_example(tokenizer, record: dict[str, object], max_length: int = max_length) -> dict[str, list[int]]:
    messages = [{"role": "user", "content": str(record["prompt"])}]
    # Qwen3 official template supports reasoning_content; keep teacher reasoning inside Thinking.
    final_content = str(record["completion"]).split("\nFinal answer:", 1)[-1]
    if record.get("trace_type") == "full_trace":
        final_content = "Final answer:" + final_content
    else:
        final_content = str(record["completion"])
    assistant = {"role": "assistant", "content": final_content, "reasoning_content":str(record.get("reasoning", ""))}
    try:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False, enable_thinking=True)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=False).input_ids
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("Chat Template 未形成可定位的 Prompt 前缀，请检查 Tokenizer 配置")
    completion_ids = full_ids[len(prompt_ids):]
    if not completion_ids:
        raise ValueError("assistant completion 为空")
    if len(full_ids) > max_length:
        raise ValueError(f"SFT trace has {len(full_ids)} tokens > MAX_LENGTH={max_length}; refuse truncated teacher completion")
    input_ids = prompt_ids + completion_ids
    prompt_length = min(len(prompt_ids), len(input_ids))
    labels = [-100] * prompt_length + input_ids[prompt_length:]
    if not any(label != -100 for label in labels):
        raise ValueError("截断后没有可训练的 Completion Token")
    return {"input_ids": input_ids, "labels": labels, "prompt_length": prompt_length}


### 4.2 Padding、Loss Sum 与有效 Token Mean

In [ ]:
def collate_causal_examples(examples: list[dict[str, list[int]]], pad_token_id: int) -> dict[str, torch.Tensor]:
    max_len = max(len(example["input_ids"]) for example in examples)
    input_ids, labels, attention_mask = [], [], []
    for example in examples:
        pad = max_len - len(example["input_ids"])
        input_ids.append(example["input_ids"] + [pad_token_id] * pad)
        labels.append(example["labels"] + [-100] * pad)
        attention_mask.append([1] * len(example["input_ids"]) + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


def masked_sft_loss_stats(logits: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, int]:
    """Return summed next-token CE and the number of valid assistant tokens."""
    if logits.ndim != 3 or labels.ndim != 2 or logits.shape[:2] != labels.shape:
        raise ValueError("logits must be [batch,time,vocab], labels must be [batch,time]")
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    valid_tokens = int(shift_labels.ne(-100).sum().item())
    if valid_tokens == 0:
        return shift_logits.sum() * 0.0, 0
    loss_sum = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100,
        reduction="sum",
    )
    return loss_sum, valid_tokens


def masked_sft_cross_entropy(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Next-token CE averaged over valid Assistant Completion Tokens only."""
    loss_sum, valid_tokens = masked_sft_loss_stats(logits, labels)
    if valid_tokens == 0:
        raise ValueError("batch has no valid Assistant Completion Token")
    return loss_sum / valid_tokens


### 4.3 自定义 SFT Epoch：按窗口有效 Token 缩放梯度

In [ ]:
class ListSFTDataset(Dataset):
    def __init__(self, encoded_examples: list[dict[str, list[int]]]):
        self.examples = encoded_examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, index): return self.examples[index]


def run_sft_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str,
    grad_accumulation_steps: int = 1,
    return_stats: bool = False,
) -> float | dict[str, object]:
    """Run one epoch with token-mean gradients in each accumulation window.

    Each microbatch contributes its summed valid-token loss. At the end of a
    window, gradients are divided by that window's actual valid-token count.
    A short final window is therefore not diluted by the configured
    accumulation length, and different Completion lengths receive token-mean
    weighting rather than an arbitrary microbatch mean.
    """
    if grad_accumulation_steps < 1:
        raise ValueError("grad_accumulation_steps must be >= 1")
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss_sum = 0.0
    total_valid_tokens = 0
    window_loss_sum = 0.0
    window_valid_tokens = 0
    window_microbatches = 0
    windows: list[dict[str, object]] = []

    def finish_window() -> None:
        nonlocal window_loss_sum, window_valid_tokens, window_microbatches
        if window_microbatches == 0:
            return
        scale = 1.0 / max(window_valid_tokens, 1)
        for parameter in model.parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(scale)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        windows.append({
            "microbatches": window_microbatches,
            "valid_tokens": window_valid_tokens,
            "loss_sum": window_loss_sum,
            "token_mean_loss": window_loss_sum / max(window_valid_tokens, 1),
        })
        optimizer.zero_grad(set_to_none=True)
        window_loss_sum = 0.0
        window_valid_tokens = 0
        window_microbatches = 0

    for step, batch in enumerate(loader):
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        logits = outputs.logits if hasattr(outputs, "logits") else outputs["logits"]
        loss_sum, valid_tokens = masked_sft_loss_stats(logits, batch["labels"])
        if valid_tokens == 0:
            continue
        loss_sum.backward()
        loss_value = float(loss_sum.detach().cpu())
        total_loss_sum += loss_value
        total_valid_tokens += valid_tokens
        window_loss_sum += loss_value
        window_valid_tokens += valid_tokens
        window_microbatches += 1
        if window_microbatches >= grad_accumulation_steps or step + 1 == len(loader):
            finish_window()
    finish_window()
    stats = {
        "mean_loss": total_loss_sum / max(total_valid_tokens, 1),
        "valid_tokens": total_valid_tokens,
        "optimizer_steps": len(windows),
        "windows": windows,
    }
    return stats if return_stats else float(stats["mean_loss"])


## 5. LoRA / QLoRA 配置与自定义训练入口

BF16 LoRA 和 NF4 QLoRA 只在 GPU 分支导入 PEFT、Transformers 和 bitsandbytes。Notebook 不安装依赖，也不在 CPU 阶段加载模型。QLoRA 只冻结 NF4 基座并训练 LoRA Adapter；激活、Adapter 梯度和优化器状态仍需显存，不能把量化理解成“所有东西都是 4-bit”。

### 5.1 LoRA 目标模块与 QLoRA 配置

In [ ]:
LORA_TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
LORA_CONFIG = {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05, "bias": "none", "target_modules": list(LORA_TARGET_MODULES)}


def build_peft_config(mode: str):
    if mode not in {"bf16_lora", "nf4_qlora"}:
        raise ValueError("mode must be bf16_lora or nf4_qlora")
    from peft import LoraConfig
    return LoraConfig(**LORA_CONFIG, task_type="CAUSAL_LM")


def build_quantization_config(mode: str):
    if mode != "nf4_qlora":
        return None
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )


def _model_load_kwargs(mode: str) -> dict[str, object]:
    if mode not in {"bf16_lora", "nf4_qlora"}:
        raise ValueError(mode)
    kwargs: dict[str, object] = {"trust_remote_code": True, "device_map": {"":0}, "torch_dtype": torch.bfloat16, "attn_implementation":"sdpa"}
    if mode == "bf16_lora":
        kwargs["torch_dtype"] = torch.bfloat16
    else:
        kwargs["quantization_config"] = build_quantization_config(mode)
    return kwargs


### 5.2 加载 Base Model，再挂载或重载 PEFT Adapter

In [ ]:
def load_sft_model_and_tokenizer(mode: str = SMOKE_QUANTIZATION_MODE, adapter_path: Path | None = None):
    """Load the base model, then create or reload a PEFT Adapter.

    `adapter_path` is never passed to AutoModelForCausalLM; it is loaded with
    PeftModel.from_pretrained after the canonical Base Model has been loaded.
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model, PeftModel, prepare_model_for_kbit_training
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
    kwargs = _model_load_kwargs(mode)
    model = AutoModelForCausalLM.from_pretrained(base_model_path, **kwargs)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if hasattr(model, "config"):
        model.config.use_cache = False
    if mode == "nf4_qlora":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    if adapter_path is None:
        model = get_peft_model(model, build_peft_config(mode))
    else:
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=True)
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    invalid = [name for name,p in model.named_parameters() if p.requires_grad and "lora_" not in name]
    if invalid: raise ValueError(f"Unexpected trainable base parameters: {invalid[:5]}")
    model.print_trainable_parameters()
    return model, tokenizer


### 5.3 Checkpoint Manifest：记录可追踪的初始化契约

In [ ]:
def _sha256_file(path: Path) -> str | None:
    import hashlib
    if not path.exists() or not path.is_file():
        return None
    return hashlib.sha256(path.read_bytes()).hexdigest()


def build_checkpoint_manifest(
    adapter_path: Path,
    mode: str,
    strategy: str,
    records: list[dict[str, object]],
    status: str,
    stage: str = "final",
) -> dict[str, object]:
    relative_adapter = str(adapter_path.relative_to(project_root)) if adapter_path.is_relative_to(project_root) else str(adapter_path)
    return {
        "schema_version": "nemotron-sft-checkpoint/v1",
        "status": status,
        "stage": stage,
        "base_model": {"id": MODEL_ID, "path": str(base_model_path)},
        "tokenizer_path": str(tokenizer_path),
        "policy_version": f"sft:{strategy}:{mode}:seed{SEED}",
        "adapter_path": relative_adapter,
        "quantization": {
            "mode": mode,
            "load_in_4bit": mode == "nf4_qlora",
            "quant_type": "nf4" if mode == "nf4_qlora" else None,
            "double_quant": mode == "nf4_qlora",
            "compute_dtype": "bfloat16" if mode == "nf4_qlora" else "bfloat16",
        },
        "lora": {**LORA_CONFIG, "task_type": "CAUSAL_LM"},
        "training": {
            "strategy": strategy,
            "data_version": _sha256_file(train_path),
            "real_trace_sha256": _sha256_file(real_trace_path),
            "use_synthetic_data": use_synthetic_data,
            "synthetic_accepted_sha256": _sha256_file(synthetic_accepted_path) if use_synthetic_data else None,
            "source_counts": dict(Counter(row.get("source", "") for row in records)),
            "records": len(records),
            "token_budget_proxy": int(sum(int(row.get("token_length_proxy", 0)) for row in records)),
            "seed": SEED,
            "epochs": NUM_EPOCHS,
            "profile": TRAINING_PROFILE,
        },
    }


def write_checkpoint_manifest(manifest: dict[str, object], path: Path = sft_manifest_path) -> Path:
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def validate_checkpoint_manifest(manifest: dict[str, object]) -> None:
    required = {"schema_version", "status", "base_model", "tokenizer_path", "adapter_path", "quantization", "lora", "training", "policy_version"}
    missing = required - set(manifest)
    if missing:
        raise ValueError(f"checkpoint manifest missing: {sorted(missing)}")
    if manifest["schema_version"] != "nemotron-sft-checkpoint/v1":
        raise ValueError("unsupported checkpoint manifest schema")
    if manifest["base_model"].get("id") != MODEL_ID:
        raise ValueError("manifest base model does not match the fixed course model")
    if manifest["quantization"].get("mode") not in {"bf16_lora", "nf4_qlora"}:
        raise ValueError("manifest quantization mode is invalid")
    if int(manifest["lora"].get("r", 0)) <= 0:
        raise ValueError("manifest LoRA rank is invalid")


### 5.4 训练数据、保存 Adapter 与单阶段调度辅助函数

In [ ]:
def _records_for_strategy(strategy: str) -> list[dict[str, object]]:
    return choose_sft_records(strategy, max_train_records)


def _train_records(model, tokenizer, records: list[dict]) -> list[dict]:
    if not records: raise ValueError("SFT stage has no records")
    encoded = [encode_chat_example(tokenizer, record) for record in records]
    token_audit = {"status":"passed", "records":len(records), "max_sequence_tokens":max(len(r["input_ids"]) for r in encoded),
        "assistant_tokens":sum(sum(label!=-100 for label in r["labels"]) for r in encoded),
        "supervised_tokens_after_shift":sum(sum(label!=-100 for label in r["labels"][1:]) for r in encoded),
        "completion_truncation":"forbidden", "source_counts":dict(Counter(row["source"] for row in records))}
    _write_stage_json(result_dir / f"tokenization_{records[0]['trace_type']}.json", token_audit)
    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    loader = DataLoader(ListSFTDataset(encoded),batch_size=MICRO_BATCH_SIZE,shuffle=True,collate_fn=lambda rows:collate_causal_examples(rows,pad))
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=LEARNING_RATE)
    history = []
    for epoch in range(NUM_EPOCHS):
        stats = run_sft_epoch(model,loader,optimizer,str(next(model.parameters()).device),GRAD_ACCUMULATION_STEPS,return_stats=True)
        if not stats["optimizer_steps"] or not stats["valid_tokens"]: raise ValueError("SFT update skipped: no supervised tokens")
        history.append({"epoch":epoch+1,"loss":stats["mean_loss"],"optimizer_steps":stats["optimizer_steps"],"valid_tokens":stats["valid_tokens"]})
    return history


def _save_adapter_checkpoint(model, tokenizer, output_dir: Path, manifest: dict[str, object], publish_global_manifest: bool = True) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    # Stage 1 is an intermediate checkpoint; only the final Stage 2 adapter
    # may become the shared SFT manifest consumed by DPO/PPO/GRPO.
    if publish_global_manifest and manifest.get("stage") not in {"stage1_recall_drill"}:
        write_checkpoint_manifest(manifest, sft_manifest_path)
    (output_dir / "checkpoint_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


### 5.5 Two-stage / Direct / Joint 训练调度

In [ ]:
def _adapter_artifact_status(adapter_path: Path) -> dict[str, object]:
    config = adapter_path / "adapter_config.json"
    weights = list(adapter_path.glob("adapter_model.*"))
    return {"path": str(adapter_path), "config_exists": config.exists(), "weights_exists": bool(weights), "reloadable_artifact": config.exists() and bool(weights)}


def _gpu_preflight(stage: str) -> dict[str, object]:
    if not torch.cuda.is_available():
        return {"status": "blocked_no_cuda", "stage": stage}
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    result = {"status": "ready", "stage": stage, "device": torch.cuda.get_device_name(0), "free_memory_mib": free_bytes / 2**20, "total_memory_mib": total_bytes / 2**20}
    result_dir.mkdir(parents=True, exist_ok=True)
    (result_dir / f"{stage}_preflight.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result


def _reload_sft_adapter(adapter_path: Path, mode: str) -> dict[str, object]:
    """Load Base + PEFT adapter again; success is required before publishing SFT manifest."""
    if not _adapter_artifact_status(adapter_path)["reloadable_artifact"]:
        return {"status": "failed_artifact_missing"}
    reloaded = None
    try:
        reloaded, _ = load_sft_model_and_tokenizer(mode, adapter_path=adapter_path)
        trainable = sum(parameter.requires_grad for parameter in reloaded.parameters())
        return {"status": "passed", "trainable_parameter_tensors": int(trainable)}
    except torch.cuda.OutOfMemoryError as exc:
        return {"status": "oom", "error": repr(exc)}
    except Exception as exc:
        return {"status": "failed", "error": repr(exc)}
    finally:
        if reloaded is not None:
            del reloaded
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_sft_training(strategy: str = "direct", mode: str = SMOKE_QUANTIZATION_MODE) -> dict[str, object]:
    if not run_sft:
        return {"status": "skipped_by_default", "stage": "sft_training", "reason": "RUN_SFT=False；GPU SFT 未请求。"}
    preflight = _gpu_preflight("sft_training")
    if preflight["status"] != "ready":
        return preflight
    if run_mode == "smoke" and max_train_records != 24:
        raise ValueError("SFT Smoke 必须使用 24 条训练轨迹")
    started = time.perf_counter()
    try:
        if strategy == "two_stage":
            stage1_records = build_recall_drill_records()
            stage1_dir = checkpoint_root / f"sft_two_stage_stage1_{mode}"
            model, tokenizer = load_sft_model_and_tokenizer(mode)
            stage1_history = _train_records(model, tokenizer, stage1_records)
            stage1_manifest = build_checkpoint_manifest(stage1_dir, mode, strategy, stage1_records, "intermediate", stage="stage1_recall_drill")
            _save_adapter_checkpoint(model, tokenizer, stage1_dir, stage1_manifest, publish_global_manifest=False)
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            stage2_records = [record for record in choose_sft_records("two_stage", max_train_records)
                              if record["trace_type"] == "full_trace"]
            model, tokenizer = load_sft_model_and_tokenizer(mode, adapter_path=stage1_dir)
            stage2_history = _train_records(model, tokenizer, stage2_records)
            all_records = stage1_records + stage2_records
            output_dir = checkpoint_root / f"sft_two_stage_{mode}"
            history = stage1_history + stage2_history
            stage = "stage2_full_trace"
        else:
            all_records = _records_for_strategy(strategy)
            model, tokenizer = load_sft_model_and_tokenizer(mode)
            history = _train_records(model, tokenizer, all_records)
            output_dir = checkpoint_root / f"sft_{strategy}_{mode}"
            stage = "final"
        manifest = build_checkpoint_manifest(output_dir, mode, strategy, all_records, "completed", stage=stage)
        _save_adapter_checkpoint(model, tokenizer, output_dir, manifest, publish_global_manifest=False)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        artifact = _adapter_artifact_status(output_dir)
        reload_check = _reload_sft_adapter(output_dir, mode)
        if not artifact["reloadable_artifact"]:
            return {"status": "failed_artifact_missing", "stage": "sft_training", "adapter_path": str(output_dir), "artifact": artifact}
        if reload_check.get("status") != "passed":
            return {"status": "failed_adapter_reload", "stage": "sft_training", "adapter_path": str(output_dir), "artifact": artifact, "adapter_reload": reload_check}
        # Publish the shared manifest only after Stage 2/final reload succeeds.
        write_checkpoint_manifest(manifest, sft_manifest_path)
        elapsed = time.perf_counter() - started
        return {"status": "completed", "stage": "sft_training", "strategy": strategy, "mode": mode, "records": len(all_records), "history": history, "training_time_seconds": elapsed, "adapter_path": str(output_dir), "manifest_path": str(sft_manifest_path), "peak_memory_allocated_mib": float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else None, "gpu_preflight": preflight, "adapter_reload": reload_check}
    except torch.cuda.OutOfMemoryError as exc:
        payload = {"status": "oom", "stage": "sft_training", "error": repr(exc), "wall_time_seconds": time.perf_counter() - started}
        (result_dir / "sft_gpu_failure.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        return payload
    except Exception as exc:
        payload = {"status": "failed", "stage": "sft_training", "error": repr(exc), "wall_time_seconds": time.perf_counter() - started}
        (result_dir / "sft_gpu_failure.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        return payload


### 5.6 GPU SFT 开关

In [ ]:
print({"gpu_sft": "deferred_to_final_execution_cell", "run_sft": run_sft})


## 6. CPU Smoke Test

使用一个极小的 Embedding + Linear 语言模型验证：Prompt Label 全为 `-100`、Completion 有效监督、Padding 不参与 Loss、反向传播和优化器更新均正常。这个测试验证算法边界，不代表 Qwen3 的训练效果、显存或吞吐。

In [ ]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size: int = 32, hidden_size: int = 16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)
    def forward(self, input_ids, attention_mask=None):
        return type("Output", (), {"logits": self.lm_head(self.embedding(input_ids))})()


def toy_encoded(prompt_len: int, completion_len: int, vocab_size: int = 32) -> dict[str, list[int]]:
    input_ids = list(range(1, prompt_len + completion_len + 1))
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    return {"input_ids": input_ids, "labels": labels, "prompt_length": prompt_len}


torch.manual_seed(SEED)
toy_examples = [toy_encoded(3, 4), toy_encoded(2, 5)]
toy_batch = collate_causal_examples(toy_examples, pad_token_id=0)
assert all(value == -100 for value in toy_batch["labels"][0, :3].tolist())
assert all(value == -100 for value in toy_batch["labels"][1, :2].tolist())
assert (toy_batch["labels"] != -100).any()

toy_model = TinyCausalLM()
toy_optimizer = torch.optim.AdamW(toy_model.parameters(), lr=1e-2)
before = [parameter.detach().clone() for parameter in toy_model.parameters()]
loss = masked_sft_cross_entropy(toy_model(toy_batch["input_ids"]).logits, toy_batch["labels"])
assert torch.isfinite(loss)
loss.backward()
toy_optimizer.step()
after = [parameter.detach().clone() for parameter in toy_model.parameters()]
assert any(not torch.equal(left, right) for left, right in zip(before, after))

# Non-divisible accumulation: 5 microbatches with accumulation=2 create
# windows [2, 2, 1]; each window uses its actual valid-token denominator.
torch.manual_seed(SEED + 1)
accum_examples = [toy_encoded(1, completion_len) for completion_len in (1, 3, 2, 4, 1)]
accum_loader = DataLoader(ListSFTDataset(accum_examples), batch_size=1, shuffle=False, collate_fn=lambda batch: collate_causal_examples(batch, 0))
accum_model = TinyCausalLM()
accum_optimizer = torch.optim.AdamW(accum_model.parameters(), lr=1e-3)
accum_stats = run_sft_epoch(accum_model, accum_loader, accum_optimizer, "cpu", grad_accumulation_steps=2, return_stats=True)
assert accum_stats["optimizer_steps"] == 3
assert [window["microbatches"] for window in accum_stats["windows"]] == [2, 2, 1]
assert len({window["valid_tokens"] for window in accum_stats["windows"]}) > 1
assert math.isfinite(float(accum_stats["mean_loss"]))

trace_audit_counts = Counter()
for record in full_trace_records[:20]:
    trace_audit_counts.update([record["trace_type"], "accepted"])

sft_cpu_report = {
    "status": "passed",
    "trace_records_checked": int(min(len(full_trace_records), 20)),
    "trace_audit": dict(trace_audit_counts),
    "label_masking": "prompt_and_padding_ignored",
    "loss_finite": True,
    "optimizer_update": True,
    "gpu_training": "not_run",
    "qlora": "configuration_only_not_run",
}
print(json.dumps(sft_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与后续实验

当前 CPU 测试只证明轨迹筛查、样本边界、Label Masking、Cross Entropy 和小模型更新正确。没有任何 GPU 训练结果，因此不能报告 Accuracy、Loss 曲线、显存或吞吐。

明天的 RTX 5090 顺序：先运行 Direct NF4 QLoRA Smoke，再运行 Joint / Two-stage；如显存允许，再执行 BF16 LoRA 对照。所有策略从同一数据版本出发，记录训练 Token Budget、训练时间、峰值显存、Adapter 保存/重载和 Validation 分任务指标。最佳 SFT Checkpoint 才能交给 DPO、PPO、GRPO。

## 7. Checkpoint Manifest 与 Adapter 交接 Smoke Test

CPU 阶段不加载 Qwen3；这里验证 SFT 交接所需的 Manifest 字段、固定 Base Model 和量化配置。GPU 阶段由 `run_sft_training` 保存 Adapter 后，再使用同一 Manifest 供 DPO/PPO/GRPO 读取。


In [ ]:
cpu_manifest = build_checkpoint_manifest(
    checkpoint_root / "cpu_fixture_adapter",
    "nf4_qlora",
    "two_stage",
    choose_sft_records("two_stage", max_train_records),
    "not_run",
)
validate_checkpoint_manifest(cpu_manifest)
assert cpu_manifest["base_model"]["id"] == MODEL_ID
assert cpu_manifest["quantization"]["load_in_4bit"] is True
assert cpu_manifest["lora"]["r"] == 8
manifest_test_path = result_dir / "cpu_checkpoint_manifest.json"
manifest_test_path.parent.mkdir(parents=True, exist_ok=True)
write_checkpoint_manifest(cpu_manifest, manifest_test_path)
assert json.loads(manifest_test_path.read_text())["schema_version"] == "nemotron-sft-checkpoint/v1"
sft_cpu_report = {"status":"passed", "smoke_records": len(choose_sft_records("two_stage", max_train_records)), "manifest": str(manifest_test_path.relative_to(project_root)), "adapter_reload": "GPU_not_run_artifact_reload_not_run", "qlora_prepare": "configuration_checked", "gradient_accumulation": {"optimizer_steps": int(accum_stats["optimizer_steps"]), "window_microbatches": [int(window["microbatches"]) for window in accum_stats["windows"]], "window_valid_tokens": [int(window["valid_tokens"]) for window in accum_stats["windows"]]}, "gpu_training": "not_run"}
(result_dir / "cpu_sft_smoke_report.json").write_text(json.dumps(sft_cpu_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(sft_cpu_report, ensure_ascii=False, indent=2))


## GPU 资源与执行状态
Stage 1 只保存中间 Adapter；最终 Adapter 重载成功后才发布共享 Manifest。实际执行入口放在所有定义与 CPU 测试之后。


In [ ]:
def _write_stage_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def _gpu_stage(stage: str, operation) -> dict:
    """Preflight + device memory sampler + truthful failure/status report."""
    import gc, threading, subprocess, traceback, importlib.metadata
    started = time.perf_counter()
    stop, samples = threading.Event(), []
    result_dir.mkdir(parents=True, exist_ok=True)
    report_path = result_dir / f"{stage}_status.json"
    if not torch.cuda.is_available():
        result = {"status": "blocked_no_cuda", "stage": stage}
        _write_stage_json(report_path, result)
        return result
    free, total = torch.cuda.mem_get_info()
    preflight = {"device": torch.cuda.get_device_name(0), "total_mib": total / 2**20,
                 "free_mib": free / 2**20, "torch_cuda": torch.version.cuda,
                 "bf16_supported": torch.cuda.is_bf16_supported()}
    preflight["versions"] = {}
    for package in ("torch", "transformers", "peft", "bitsandbytes", "vllm"):
        try: preflight["versions"][package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: pass
    _write_stage_json(result_dir / f"{stage}_preflight.json", preflight)
    def monitor():
        while not stop.is_set():
            try:
                output = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True, timeout=3)
                samples.append(float(output.splitlines()[0]))
            except (OSError, ValueError, subprocess.SubprocessError): pass
            stop.wait(0.5)
    thread = threading.Thread(target=monitor, daemon=True)
    torch.cuda.reset_peak_memory_stats()
    thread.start()
    try:
        if "training" in stage and "vllm" in __import__("sys").modules:
            result = {"status": "blocked_kernel_switch_required", "reason": "先结束 vLLM 进程并切换训练 Kernel"}
        elif free / total < 0.80:
            result = {"status": "blocked_gpu_occupied", "reason": "GPU 可用显存不足 80%；不终止其他任务"}
        else:
            result = operation()
            if not isinstance(result, dict) or "status" not in result:
                raise ValueError("阶段必须显式返回 status")
    except Exception as exc:
        message = traceback.format_exc()
        result = {"status": "oom" if isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower() else "failed",
                  "error": repr(exc), "traceback": message}
        (result_dir / f"{stage}_error.log").write_text(message, encoding="utf-8")
    finally:
        stop.set(); thread.join(timeout=4)
        gc.collect(); torch.cuda.empty_cache()
    result.update({"stage": stage, "wall_time_seconds": time.perf_counter() - started,
                   "preflight": preflight, "peak_memory_allocated_mib": torch.cuda.max_memory_allocated() / 2**20,
                   "peak_device_memory_mib": max(samples) if samples else None})
    _write_stage_json(report_path, result)
    return result


### CPU 检查：Two-stage 的固定预算与类别覆盖
Recall Drill 与 Full Trace 合计仍为 24 条；预留 Recall 预算之后，Full Trace 再次分层，确保三种任务都参与 Stage 2。

In [ ]:
_two_stage_records = choose_sft_records("two_stage", 24)
_stage2_counts = Counter(r["task_type"] for r in _two_stage_records if r["trace_type"] == "full_trace")
assert len(_two_stage_records) == 24
assert _stage2_counts == {"gravity": 5, "equation_numeric": 5, "bit_manipulation": 5}
print({"two_stage_full_trace_counts": dict(_stage2_counts), "status": "passed"})


In [ ]:
if run_sft:
    sft_status = _gpu_stage("sft_training", lambda: run_sft_training(os.environ.get("SFT_STRATEGY", "two_stage"), SMOKE_QUANTIZATION_MODE))
    print(json.dumps(sft_status,ensure_ascii=False,indent=2))
else:
    sft_status = {"status":"skipped_by_default"}


### 长序列低显存 SFT
逐块完整词表交叉熵；只对 assistant token 求和，再按累积窗口 token 数归一化。


In [ ]:
def course_sft_loss_sum(model, batch, chunk_size=128):
    from utils.low_memory_logprobs import causal_policy_logprobs
    labels = batch["labels"][:, 1:]
    active = labels.ne(-100)
    logp = causal_policy_logprobs(model, batch["input_ids"], batch["attention_mask"], chunk_size)
    return -(logp * active).sum(), int(active.sum())
